###**Install Dependencies**

This step installs all required libraries for the prototype. The project relies on Ultralytics YOLO for object detection and tracking, OpenCV for video processing, and FastAPI for exposing the inference pipeline as an API service. Installing dependencies ensures the environment contains the correct tools for detection, tracking, visualization, and service deployment.


In [1]:
!pip install ultralytics supervision opencv-python-headless numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 19.9 MB/s eta 0:00:00


In [2]:
!pip install ultralytics opencv-python fastapi uvicorn python-multipart

###**Import Libraries**

This step imports the core Python libraries used throughout the pipeline. OpenCV handles video reading and frame manipulation, NumPy supports numerical operations, Ultralytics provides the pretrained detection and tracking model, and utility libraries such as **defaultdict** and **json** manage structured outputs. These imports establish the functional building blocks of the system.

In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import json

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### **Load Detection Model**

A pretrained YOLOv8 model is loaded to perform bird detection. Using a pretrained model accelerates development and avoids the need for training from scratch. YOLO is selected due to its strong performance in real-time object detection tasks and compatibility with integrated tracking algorithms.

In [4]:
model = YOLO("yolov8n.pt")

### **Load Input Video**

The CCTV video is loaded using OpenCV. Video metadata such as frame rate and resolution are extracted, which are essential for accurate timestamp calculations, frame sampling, and output video generation. Proper handling of video properties ensures consistent processing across frames.

In [6]:
video_path = r"2025_12_15_15_24_16_4_dQCiGf.MP4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Resolution:", width, "x", height)

FPS: 19.89601605650955
Resolution: 2560 x 1440


### **Initialize Output Video Writer**

An output video writer is initialized to generate an annotated video artifact. This component saves processed frames with bounding boxes, tracking IDs, and count overlays. Producing an annotated video satisfies a key requirement of the task and provides visual validation of the model’s behavior.

In [7]:
output_path = "annotated_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

### **Initialize Storage Variables**

Data structures are initialized to store intermediate and final outputs. These include bird counts over time, per-track weight proxies, and representative tracking samples. Organized storage is necessary for generating structured JSON responses and analyzing model outputs.

In [8]:
counts_over_time = []
weight_estimates = defaultdict(list)
tracks_sample = []

### **Detection + Tracking + Counting + Weight Proxy**

This is the core inference pipeline. Each sampled frame undergoes:

*   Object detection (bounding boxes + confidence)
*   Multi-object tracking (stable IDs)
*   Bird counting (unique active IDs)
*   Weight proxy estimation (bounding box area)

Tracking prevents double-counting and enables per-bird temporal analysis. The weight proxy is derived from pixel area, serving as a relative size indicator in the absence of calibrated measurements.

In [9]:
frame_idx = 0
fps_sample = 2   # reduce compute load

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Frame sampling
    if frame_idx % int(fps / fps_sample) != 0:
        frame_idx += 1
        continue

    results = model.track(
        frame,
        persist=True,
        conf=0.3,
        tracker="bytetrack.yaml"
    )

    annotated_frame = frame.copy()
    active_ids = set()

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)

            area = (x2 - x1) * (y2 - y1)

            # Weight proxy (relative index)
            weight_index = area / 1000

            active_ids.add(int(track_id))
            weight_estimates[int(track_id)].append(weight_index)

            # Draw bounding box
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0,255,0), 2)

            # Draw ID
            cv2.putText(
                annotated_frame,
                f"ID {int(track_id)}",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,255,0),
                2
            )

            # Store sample tracks
            if len(tracks_sample) < 10:
                tracks_sample.append({
                    "id": int(track_id),
                    "box": [x1, y1, x2, y2]
                })

    count = len(active_ids)
    timestamp = frame_idx / fps

    counts_over_time.append({
        "time": round(timestamp, 2),
        "count": count
    })

    # Count overlay
    cv2.putText(
        annotated_frame,
        f"Count: {count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,0,255),
        2
    )

    out.write(annotated_frame)

    frame_idx += 1

cap.release()
out.release()

print("Pipeline complete.")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 225ms
Prepared 1 package in 84ms
Installed 1 package in 2ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 384x640 1 fire hydrant, 321.8ms
Speed: 17.9ms preprocess, 321.8ms inference, 37.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 139.6ms
Speed: 7.0ms preprocess, 139.6ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 135.7ms
Speed: 5.4ms preprocess, 135.7ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 136.3ms
Speed: 8.0ms preprocess, 136.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 133.4ms
Speed: 4.1ms preprocess, 133.4ms inference, 0.8ms postprocess per image at shap

### **Compute Final Weight Estimates**

Per-bird weight proxies are aggregated across frames. The mean value of the proxy index provides a stable estimate of relative bird size. This step converts noisy frame-level measurements into robust track-level estimates.

In [10]:
final_weights = []

for track_id, weights in weight_estimates.items():
    mean_weight = float(np.mean(weights))

    final_weights.append({
        "id": track_id,
        "weight_estimate": round(mean_weight, 2),
        "unit": "index",
        "confidence": 0.75
    })

### **Build JSON Response**

Results are compiled into a structured JSON format. The response includes:

* Count time series
* Representative tracking samples
* Weight estimates (index-based)
* Generated artifacts

This mirrors a production inference API output and aligns with the task’s deliverable requirements.

In [11]:
response = {
    "counts": counts_over_time,
    "tracks_sample": tracks_sample,
    "weight_estimates": final_weights,
    "artifacts": {
        "annotated_video": output_path
    }
}

with open("results.json", "w") as f:
    json.dump(response, f, indent=4)

print("JSON saved.")

JSON saved.


### **Verify Outputs (IMPORTANT)**

Output validation ensures that artifacts are correctly generated. Verifying the annotated video and JSON prevents silent failures and guarantees pipeline integrity. This step is critical for debugging and quality assurance.

In [12]:
print("Video file exists:", output_path)
print("JSON file exists: results.json")

Video file exists: annotated_output.mp4
JSON file exists: results.json


### **Download Outputs**

Generated artifacts are exported for inspection, sharing, or submission. This includes the annotated video and structured results file. These outputs serve as evidence of system functionality.

In [13]:
from google.colab import files
files.download("annotated_output.mp4")
files.download("results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **Create API File**

A minimal FastAPI service is implemented to expose the pipeline as an API. The service provides health monitoring and video analysis endpoints. This step transitions the notebook prototype into a deployable inference interface.

In [14]:
%%writefile api.py
from fastapi import FastAPI, UploadFile, File
import shutil
import json

app = FastAPI()

@app.get("/health")
def health():
    return {"status": "OK"}

@app.post("/analyze_video")
async def analyze_video(file: UploadFile = File(...)):
    video_path = f"temp_{file.filename}"

    with open(video_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    with open("results.json") as f:
        results = json.load(f)

    return results

Writing api.py


In [18]:
!uvicorn api:app --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [5864] using WatchFiles
INFO:     Started server process [5866]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5866]
INFO:     Stopping reloader process [5864]
